# Silver Layer


In [0]:
bronze_df = spark.read.table("bronze.fhir_raw")

In [0]:
bronze_df.printSchema()

one row contain array of struct so we will explode the data so that one row contain one struct only

In [0]:
from pyspark.sql.functions import explode

In [0]:
resource_df = bronze_df.select(explode("entry").alias("entry"))

In [0]:
# resource_df.groupBy(
#     "entry.resource.resourceType"
# ).count().show()

## Patient Record

In [0]:
patient_df = resource_df.filter(resource_df["entry.resource.resourceType"] == 'Patient')

In [0]:
# patient_df.printSchema()

In [0]:
from pyspark.sql.functions import col

patient_silver_df = patient_df.select(
    col("entry.resource.id").alias("Patient_Id"),
    col("entry.resource.birthDate").alias("Birth_date"),
    col("entry.resource.gender").alias("Gender"),
    col("entry.resource.name").alias("Name")
)

display(patient_silver_df)


In [0]:
from pyspark.sql.functions import get_json_object

In [0]:
patient_silver_df = patient_silver_df.withColumn(
    "given_name",
    get_json_object("name", "$[0].given[0]")
).withColumn(
    "family_name",
    get_json_object("name", "$[0].family")
)

In [0]:

patient_silver_df.select(
    "given_name",
    "family_name"
).show(5, truncate=False)



In [0]:
patient_silver_df = patient_silver_df.drop(
    "name"
)

In [0]:
from pyspark.sql.functions import concat_ws

patient_silver_df = patient_silver_df.withColumn(
    "full_name",
    concat_ws(" ", "given_name", "family_name")
)

In [0]:
display(patient_silver_df)

## Encourter Records

In [0]:
encounter_df = resource_df.filter(resource_df["entry.resource.resourceType"] == 'Encounter')

encounter_silver_df= encounter_df.select(
    col("entry.resource.id").alias("Encounter_Id"),
    col("entry.resource.status").alias("Status"),
    col("entry.resource.class.code").alias("Class"),
    col("entry.resource.subject.reference").alias("Patient_Reference"),
    col("entry.resource.subject.display").alias("Patient_Name"),
    col("entry.resource.period.start").alias("Start_Date"),
    col("entry.resource.period.end").alias("End_Date")
)

In [0]:
display(encounter_silver_df)

## Condition Records

In [0]:
from pyspark.sql.functions import get_json_object

In [0]:
condition_df = resource_df.filter(resource_df["entry.resource.resourceType"]=="Condition")

condition_silver_df = condition_df.select(
    col("entry.resource.id").alias("Condition_Id"),
    col("entry.resource.clinicalStatus.coding.code")[0].alias("Clinical_Status"),
    col("entry.resource.verificationStatus.coding.code")[0].alias("Verification_Status"),
    get_json_object(
        col("entry.resource.code"),
        "$.coding[0].code"
    ).alias("Condition_Code"),

   
    get_json_object(
        col("entry.resource.code"),
        "$.coding[0].display"
    ).alias("Condition_Name"),

    get_json_object(
        col("entry.resource.category")[0],
        "$.coding[0].display"
    ).alias("Category"),
   
    col("entry.resource.subject.reference").alias("Patient_Reference"),
    col("entry.resource.encounter.reference").alias("Encounter_Reference"),
    col("entry.resource.recordedDate").alias("Recorded_Date"),
    col("entry.resource.onsetDateTime").alias("Onset_Date")
)

display(condition_silver_df)



## Observation Resource

In [0]:
observation_df = resource_df.filter(resource_df["entry.resource.resourceType"]=="Observation")



In [0]:
from pyspark.sql.functions import to_date

observation_silver_df = observation_df.select(
    col("entry.resource.id").alias("Observation_Id"),
    col("entry.resource.status").alias("Status"),

    get_json_object(
        col("entry.resource.category")[0],
        "$.coding[0].display"
    ).alias("Observation_Name"),

    col("entry.resource.subject.reference").alias("Patient_Reference"),
    col("entry.resource.encounter.reference").alias("Encounter_Reference"),
    to_date(col("entry.resource.effectiveDateTime")).alias("Observation_Date"),
    to_date(col("entry.resource.issued")).alias("Issued_Date"),
    col("entry.resource.valueQuantity.value").alias("Value"),
    col("entry.resource.valueQuantity.unit").alias("Unit"),

    

    
)
display(observation_silver_df)

## Medication Request Resource

In [0]:
medicationRequest_df = resource_df.filter(resource_df["entry.resource.resourceType"]=="MedicationRequest")

In [0]:
medicationRequest_silver_df = medicationRequest_df.select(
    col("entry.resource.id").alias("MedicationRequest_Id"),
    col("entry.resource.status").alias("Status"),
    col("entry.resource.intent").alias("Intent"),
    col("entry.resource.subject.reference").alias("Patient_Reference"),
    col("entry.resource.medicationCodeableConcept.coding.code")[0].alias("Medication_Code"),
    col("entry.resource.medicationCodeableConcept.coding.display")[0].alias("Medication_Name"),
    col("entry.resource.encounter.reference").alias("Encounter_Reference"),
    col("entry.resource.authoredOn").alias("Authored_On"),
    col("entry.resource.requester.reference").alias("Requester_Reference"),
    col("entry.resource.requester.display").alias("Requester_Name"),
    
)


display(medicationRequest_silver_df)

##Deduplication logic using Resource IDs as surrogate key

In [0]:
patient_silver_df = patient_silver_df.dropDuplicates(["Patient_ID"])
encounter_silver_df = encounter_silver_df.dropDuplicates(["Encounter_ID"])
condition_silver_df = condition_silver_df.dropDuplicates(["Condition_ID"])
observation_silver_df = observation_silver_df.dropDuplicates(["Observation_ID"])
medicationRequest_silver_df = medicationRequest_silver_df.dropDuplicates(["MedicationRequest_ID"])

## Delta Live Tables

In [0]:
from pyspark import pipelines as dp

@dp.table(name="patient_silver_dlt")
@dp.expect("valid_patient_id", "Patient_Id IS NOT NULL")
@dp.expect("valid_birth_date", "Birth_date IS NOT NULL")
@dp.expect("valid_gender", "Gender IN ('male','female')")
def patient_silver():
    return patient_silver_df


@dp.table(name="encounter_silver_dlt")
@dp.expect("valid_encounter_id", "Encounter_Id IS NOT NULL")
@dp.expect("valid_patient_reference", "Patient_Reference IS NOT NULL")
@dp.expect("valid_encounter_status", "Status IS NOT NULL")
def encounter_silver():
    return encounter_silver_df


@dp.table(name="condition_silver_dlt")
@dp.expect("valid_condition_id", "Condition_Id IS NOT NULL")
@dp.expect("valid_patient_reference", "Patient_Reference IS NOT NULL")
@dp.expect("valid_condition_name", "Condition_Name IS NOT NULL")
def condition_silver():
    return condition_silver_df


@dp.table(name="observation_silver_dlt")
@dp.expect("valid_observation_id", "Observation_Id IS NOT NULL")
@dp.expect("valid_patient_reference", "Patient_Reference IS NOT NULL")
@dp.expect("valid_status", "Status IS NOT NULL")

def observation_silver():
    return observation_silver_df


@dp.table(name="medication_request_silver_dlt")
@dp.expect("valid_medication_request_id", "MedicationRequest_Id IS NOT NULL")
@dp.expect("valid_medication_name", "Medication_Name IS NOT NULL")
@dp.expect("valid_patient_reference", "Patient_Reference IS NOT NULL")

def medication_request_silver():
    return medicationRequest_silver_df

DLT data quality expectations identified 354 MedicationRequest records with missing Medication_Name values (1.5% of records). These records were logged as quality violations while still being retained in the Silver layer.